# FMS Rule Generator - Model Training
This notebook generates synthetic transaction data and trains a decision tree model to generate fraud detection rules.

In [ ]:
# Install required packages
!pip install scikit-learn pandas numpy matplotlib joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import matplotlib.pyplot as plt
from sklearn.tree import _tree

## Step 1: Generate Synthetic Transaction Data

In [ ]:
np.random.seed(42)
n_samples = 10000

amounts = np.random.uniform(10, 10000, n_samples)
transaction_fees = amounts * np.random.uniform(0.01, 0.05, n_samples)
settlement_fees = amounts * np.random.uniform(0.005, 0.02, n_samples)
transaction_processing_fees = np.random.uniform(0.5, 5, n_samples)
settlement_processing_fees = np.random.uniform(0.3, 3, n_samples)

rules = []
for i in range(n_samples):
    amt = amounts[i]
    txn_fee = transaction_fees[i]
    
    if amt > 5000:
        rules.append('high_amount_transaction')
    elif amt > 2000 and txn_fee > 50:
        rules.append('medium_high_risk')
    elif amt > 1000:
        rules.append('medium_amount_transaction')
    elif amt < 100 and txn_fee < 2:
        rules.append('micro_transaction')
    elif txn_fee > amt * 0.04:
        rules.append('high_fee_transaction')
    elif settlement_fees[i] > 20:
        rules.append('high_settlement_cost')
    elif amt > 500 and amt <= 1000:
        rules.append('standard_transaction')
    else:
        rules.append('low_risk_transaction')

df = pd.DataFrame({
    'amount': amounts,
    'transactionFeeAmount': transaction_fees,
    'settlementFeeAmount': settlement_fees,
    'transactionProcessingFee': transaction_processing_fees,
    'settlementProcessingFee': settlement_processing_fees,
    'fired_rule_name': rules
})

print(f'Generated {len(df)} transactions')
print(f'Unique rules: {df["fired_rule_name"].nunique()}')
print(f'\nRule distribution:\n{df["fired_rule_name"].value_counts()}')

## Step 2: Train Decision Tree Model

In [ ]:
feature_columns = ['amount', 'transactionFeeAmount', 'settlementFeeAmount',
                   'transactionProcessingFee', 'settlementProcessingFee']

X = df[feature_columns]
y = df['fired_rule_name']

y_encoder = LabelEncoder()
y_encoded = y_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)

print(f'Model trained! Tree depth: {clf.get_depth()}, Leaves: {clf.get_n_leaves()}')

y_pred = clf.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}')

## Step 3: Visualize Decision Tree

In [ ]:
plt.figure(figsize=(25,15))
plot_tree(clf, feature_names=feature_columns, class_names=y_encoder.classes_, 
          filled=True, fontsize=8, max_depth=3)
plt.savefig('decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4: Generate Drools Rules

In [ ]:
def generate_drools_rules(clf, y_encoder, feature_names):
    tree_ = clf.tree_
    rules = []
    rule_counter = [0]
    
    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_names[tree_.feature[node]]
            threshold = tree_.threshold[node]
            left_cond = conditions + [f'{name} > {threshold:.2f}']
            right_cond = conditions + [f'{name} <= {threshold:.2f}']
            recurse(tree_.children_left[node], right_cond)
            recurse(tree_.children_right[node], left_cond)
        else:
            if not conditions:
                return
            
            value = tree_.value[node][0]
            class_idx = value.argmax()
            rule_name = y_encoder.inverse_transform([class_idx])[0]
            conditions_str = ' , '.join(conditions)
            rule_id = f'AI_Generated_{rule_name}_{rule_counter[0]}'
            rule_counter[0] += 1

            drools_rule = f'''import net.com.fms_core.dto.message.IsoMessageDTO;

rule "{rule_id}"
when
    $t : IsoMessageDTO({conditions_str})
then
    $t.setRiskLevel("HIGH");
    $t.setFlaggedForReview(true);
    $t.setRuleFired(true);
    $t.setRuleName("{rule_name}");
    $t.setFiredRule("Generated by AI");
    $t.setRiskScore(7.0);
end
'''
            rules.append(drools_rule)
    
    recurse(0, [])
    return rules

rules = generate_drools_rules(clf, y_encoder, feature_columns)
print(f'Generated {len(rules)} Drools rules\n')
print('Preview of first 2 rules:')
print('='*80)
for rule in rules[:2]:
    print(rule)

## Step 5: Save Model and Rules

In [ ]:
# Save model
joblib.dump({'model': clf, 'encoder': y_encoder}, 'decision_tree_model.pkl')
print('✓ Model saved to decision_tree_model.pkl')

# Save rules
with open('generated_rules.drl', 'w') as f:
    for rule in rules:
        f.write(rule + '\n\n')
print('✓ Rules saved to generated_rules.drl')

# Save dataset
df.to_csv('synthetic_transaction_history.csv', index=False)
print('✓ Dataset saved to synthetic_transaction_history.csv')

print('\n' + '='*80)
print('COMPLETE! Download decision_tree_model.pkl and place it in your rulegenerator folder')
print('='*80)